# Customer Support Ticket Router

A small LangGraph example that classifies a support ticket with keyword rules and routes it to a billing or technical reply node. A reducer accumulates the nodes visited on each path. No LLM or API key is needed.

## Setup

Install LangGraph in the Colab runtime. If the runtime asks for it, restart the session after installation.

In [ ]:
%pip -q install langgraph

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: str
    reply: str
    path_history: Annotated[list[str], operator.add]


def classify_ticket(state: TicketState) -> dict[str, object]:
    """Choose a category using simple, case-insensitive keyword rules."""
    text = state["ticket"].lower()
    billing_keywords = ("bill", "charge", "payment", "refund", "invoice")
    category = "billing" if any(word in text for word in billing_keywords) else "technical"
    return {"category": category, "path_history": ["classify"]}


def route_ticket(state: TicketState) -> Literal["billing", "technical"]:
    return state["category"]


def billing_reply(state: TicketState) -> dict[str, object]:
    return {
        "reply": "I can help with your billing question. Please share the invoice or charge details so we can look into it.",
        "path_history": ["billing"],
    }


def technical_reply(state: TicketState) -> dict[str, object]:
    return {
        "reply": "I can help troubleshoot this technical issue. Please share the steps to reproduce it and any error message.",
        "path_history": ["technical"],
    }

## Build the graph

The classify node records its visit and chooses a category. A conditional edge sends the ticket to the matching reply node, which appends its own visit to the history.

In [ ]:
builder = StateGraph(TicketState)
builder.add_node("classify", classify_ticket)
builder.add_node("billing", billing_reply)
builder.add_node("technical", technical_reply)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_ticket)
builder.add_edge("billing", END)
builder.add_edge("technical", END)

ticket_router = builder.compile()

## Draw the graph

Render the compiled graph as an image. LangGraph may install its Mermaid rendering dependencies on the first call.

In [ ]:
display(Image(ticket_router.get_graph().draw_mermaid_png()))

## Try one ticket per route

In [ ]:
sample_tickets = [
    "I was charged twice for my monthly subscription.",
    "The app crashes whenever I try to upload a file.",
]

for ticket in sample_tickets:
    result = ticket_router.invoke(
        {"ticket": ticket, "category": "", "reply": "", "path_history": []}
    )
    print(f"Ticket:   {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply:    {result['reply']}")
    print(f"Path:     {' -> '.join(result['path_history'])}\n")